In [29]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from experiment_configs import get_experiment_dict

In [30]:
class TCCRMSEStatistics:
    def __init__(self, outdir, experiments_dict, varnames, windows,
                 landmask, metrics=['TCC', 'RMSE'], percentiles=[0.9],
                 nens_key='nens', save_outdir=None):
        """
        Parameters:
        - percentiles: list of percentiles (e.g., [0.95, 0.9, 0.75, 0.5, 0.25])
        """
        self.outdir = outdir
        self.experiments_dict = experiments_dict
        self.varnames = varnames
        self.windows = windows
        self.landmask = landmask
        self.metrics = metrics
        self.percentiles = percentiles
        self.nens_key = nens_key
        self.save_outdir = save_outdir or os.path.join(outdir, "fixed_summary_netcdf")
        os.makedirs(self.save_outdir, exist_ok=True)

    def _load_metric(self, exp_name, varname, window_label, nens, metric):
        paths = [
            os.path.join(self.outdir, exp_name, f"fcst_tcc_rmse_{exp_name}_EN{n:02d}_{varname}_{window_label}.nc")
            for n in range(1, nens + 1)
        ]
        da_list = []
        for p in paths:
            if os.path.exists(p):
                ds = xr.open_dataset(p)
                if metric in ds:
                    da_list.append(ds[metric])
        if not da_list:
            raise FileNotFoundError(f"[ERROR] Missing {metric} for {exp_name} {varname} {window_label}")
        return xr.concat(da_list, dim='ensemble')

    def _region_mean(self, da, region):
        if self.landmask is not None:
            if region == "land":
                da = da.where(self.landmask >= 0.5)
            elif region == "ocean":
                da = da.where(self.landmask < 0.5)
    
        # Clean non-finite values (NaN, +inf, -inf)
        da = da.where(np.isfinite(da))
    
        if da.notnull().sum() == 0:
            return xr.DataArray(np.nan)
    
        weights = np.cos(np.deg2rad(da.lat))
        return da.weighted(weights).mean(("lat", "lon"), skipna=True)
    
    def process_experiments(self, region='global'):
        for exp_name, meta in self.experiments_dict.items():
            print(f"\n[INFO] Processing {exp_name}")
            nens = meta[self.nens_key]
            data_vars = {}

            for metric in self.metrics:
                for pct in self.percentiles:
                    pct_str = f"{int(pct * 100)}"
                    df = pd.DataFrame(index=self.windows.keys(), columns=self.varnames.keys())

                    for win_label in self.windows:
                        for var in self.varnames:
                            try:
                                ens_da = self._load_metric(exp_name, var, win_label, nens, metric)
                                # Clean all non-finite values: NaN, -inf, +inf
                                ens_da = ens_da.where(np.isfinite(ens_da))
                                # Drop ensemble members that are fully NaN
                                ens_da = ens_da.where(ens_da.notnull().any(dim=("lat", "lon")), drop=True)
                                # Mask grid points with no valid values across ensemble
                                ens_da = ens_da.where(ens_da.notnull().any(dim="ensemble"))

                                # Mask out grid points with no valid ensemble values
                                valid_mask = ens_da.notnull().any(dim="ensemble")
                                ens_da = ens_da.where(valid_mask)
                            
                                if ens_da.sizes.get("ensemble", 0) < 2:
                                    raise ValueError("Too few valid ensemble members to compute quantile.")
                            
                                # Compute quantile safely
                                pval = ens_da.quantile(pct, dim='ensemble', skipna=True)
                            
                                # Spatial average
                                mean_val = self._region_mean(pval, region).item()
                                df.loc[win_label, var] = mean_val
                            
                            except Exception as e:
                                print(f"[WARNING] {metric}_p{pct_str} | {exp_name} | {var} | {win_label} skipped: {e}")
                                df.loc[win_label, var] = np.nan
                    da = xr.DataArray(
                        df.astype(float).values,
                        coords={"window": list(df.index), "variable": list(df.columns)},
                        dims=["window", "variable"],
                        name=f"{metric}_p{pct_str}"
                    )
                    da.attrs.update({
                        "metric": metric,
                        "percentile": pct,
                        "region": region
                    })
                    data_vars[da.name] = da

            ds = xr.Dataset(data_vars)
            out_path = os.path.join(
                self.save_outdir,
                f"tcc_rmse_metrics_allpct_{region}_{exp_name}.nc"
            )
            ds.to_netcdf(out_path)
            print(f"[SAVED] {out_path}")

    def check_and_repair_variable(self, exp_name, varname="PRECT", region="global",
                                  percentiles=None, nan_threshold=0.5):
        if percentiles is None:
            percentiles = self.percentiles
    
        summary_file = os.path.join(
            self.save_outdir,
            f"tcc_rmse_metrics_allpct_{region}_{exp_name}.nc"
        )
    
        if not os.path.exists(summary_file):
            print(f"[ERROR] Summary file not found: {summary_file}")
            return
    
        ds = xr.open_dataset(summary_file)
        print(f"[INFO] Loaded summary: {summary_file}")
    
        nens = self.experiments_dict[exp_name][self.nens_key]
    
        for metric in self.metrics:
            for pct in percentiles:
                pct_str = f"p{int(round(pct * 100))}"
                varname_index = ds['variable'].values.tolist().index(varname)
                data_var_name = f"{metric}_{pct_str}"
    
                if data_var_name not in ds:
                    print(f"[WARNING] {data_var_name} not found in dataset. Skipping.")
                    continue
    
                da = ds[data_var_name]
                column = da.isel(variable=varname_index)

                nan_frac = column.isnull().sum().item() / column.size
                outlier = (np.abs(column) > 1e5).any().item()
    
                if nan_frac < nan_threshold and not outlier:
                    print(f"[OK] {data_var_name}[{varname}] is valid in {exp_name}")
                    continue
                else:
                    print(f"[FIXING] {data_var_name}[{varname}] has nan_frac={nan_frac:.2f}, outlier={outlier}")
    
                # Repair: recompute new column values
                repaired_vals = []
                for win_label in self.windows:
                    try:
                        ens_da = self._load_metric(exp_name, varname, win_label, nens, metric)
                        pval = ens_da.quantile(pct, dim='ensemble', skipna=True)
                        mean_val = self._region_mean(pval, region).item()
                        repaired_vals.append(mean_val)
                    except Exception as e:
                        print(f"[SKIPPED] Cannot compute {data_var_name} for {win_label}: {e}")
                        repaired_vals.append(np.nan)
    
                # Replace the PRECT column in place
                da.loc[dict(variable=varname)] = xr.DataArray(
                    repaired_vals, coords={"window": list(self.windows.keys())}, dims="window"
                )
                print(f"[REPLACED] {data_var_name}[{varname}] in {exp_name}")
    
        # Save back the updated dataset
        ds.to_netcdf(summary_file, mode='w')
        print(f"[SAVED] Updated file: {summary_file}")
        

In [31]:
if __name__ == "__main__":
    top_path = "/pscratch/sd/z/zhan391/e3sm_dart"
    out_path = os.path.join(top_path, "diag_dart")
    landmask_file = os.path.join(out_path, "landmask_1x1.nc")
    fig_path = "/global/homes/z/zhan391/analysis/diagnostic/figures"

    years = [2012]
    exp = 'v3_hindcast'
    exp_dict = get_experiment_dict(exp)  # Should return experiment configs with 'nens'

    # Map of experiment names to their summary NetCDF files
    summary_file_dict = {
        "CTRLEN10": f"{out_path}/tcc_rmse_metrics_allpct_global_CTRLEN10.nc",
        "CAPTEN20": f"{out_path}/tcc_rmse_metrics_allpct_global_CAPTEN10.nc",
        "DARTEN20": f"{out_path}/tcc_rmse_metrics_allpct_global_DARTEN40.nc",
        "DARTEN40": f"{out_path}/tcc_rmse_metrics_allpct_global_DARTEN40.nc"
    }
    

    windows = {
        "D01-15":   ("2012-01-01", "2012-01-15"),
        "D16-30":   ("2012-01-16", "2012-01-30"),
        "D31-45":   ("2012-01-31", "2012-02-14"),
        "D46-60":   ("2012-02-15", "2012-03-01"),
    }

    varnames = {'PRECT': 'm/s'}
    
    # Load landmask
    landmask = xr.open_dataset(landmask_file)['landmask']

    # Initialize evaluator for multiple percentiles
    evaluator = TCCRMSEStatistics(
        outdir=out_path,
        experiments_dict=exp_dict,
        varnames=varnames,
        windows=windows,
        landmask=landmask,
        metrics=['TCC', 'RMSE'],
        percentiles=[0.95, 0.90, 0.75, 0.50, 0.25],
        save_outdir=os.path.join(out_path, "fixed_metrics_netcdf")
    )

    # Run evaluation
    l_fix_bad_values = False 
    if l_fix_bad_values: 
        #############################################################################################
        #comment out if only need to repair certain variable 
        varfixes = {'PRECT': 'm/s'}
        # Step 2: Post-check and repair PRECT for experiments with known issues
        for varname in varfixes:
            for exp_to_fix in ["DARTEN20", "DARTEN40"]:
                evaluator.check_and_repair_variable(exp_name=exp_to_fix, varname=varname, region='land')
        ############################################################################################ 
    else: 
        # Run evaluation
        evaluator.process_experiments(region='global')  # or 'land', 'ocean'


[INFO] Processing CTRLEN10


/global/common/software/e3sm/anaconda_envs/base/envs/e3sm_unified_1.10.0_login/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/global/common/software/e3sm/anaconda_envs/base/envs/e3sm_unified_1.10.0_login/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/global/common/software/e3sm/anaconda_envs/base/envs/e3sm_unified_1.10.0_login/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/global/common/software/e3sm/anaconda_envs/base/envs/e3sm_unified_1.10.0_login/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/global/common/software/e3sm/anaconda_envs/base/envs/e3sm_unified_1.10.0_login/lib/python3.10/site-packages/numpy/lib/nanfunctions.p

[SAVED] /pscratch/sd/z/zhan391/e3sm_dart/diag_dart/fixed_metrics_netcdf/tcc_rmse_metrics_allpct_global_CTRLEN10.nc

[INFO] Processing CAPTEN10


/global/common/software/e3sm/anaconda_envs/base/envs/e3sm_unified_1.10.0_login/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/global/common/software/e3sm/anaconda_envs/base/envs/e3sm_unified_1.10.0_login/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/global/common/software/e3sm/anaconda_envs/base/envs/e3sm_unified_1.10.0_login/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/global/common/software/e3sm/anaconda_envs/base/envs/e3sm_unified_1.10.0_login/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/global/common/software/e3sm/anaconda_envs/base/envs/e3sm_unified_1.10.0_login/lib/python3.10/site-packages/numpy/lib/nanfunctions.p

[SAVED] /pscratch/sd/z/zhan391/e3sm_dart/diag_dart/fixed_metrics_netcdf/tcc_rmse_metrics_allpct_global_CAPTEN10.nc

[INFO] Processing DARTEN20


/global/common/software/e3sm/anaconda_envs/base/envs/e3sm_unified_1.10.0_login/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/global/common/software/e3sm/anaconda_envs/base/envs/e3sm_unified_1.10.0_login/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/global/common/software/e3sm/anaconda_envs/base/envs/e3sm_unified_1.10.0_login/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/global/common/software/e3sm/anaconda_envs/base/envs/e3sm_unified_1.10.0_login/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/global/common/software/e3sm/anaconda_envs/base/envs/e3sm_unified_1.10.0_login/lib/python3.10/site-packages/numpy/lib/nanfunctions.p

[SAVED] /pscratch/sd/z/zhan391/e3sm_dart/diag_dart/fixed_metrics_netcdf/tcc_rmse_metrics_allpct_global_DARTEN20.nc

[INFO] Processing DARTEN40


/global/common/software/e3sm/anaconda_envs/base/envs/e3sm_unified_1.10.0_login/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/global/common/software/e3sm/anaconda_envs/base/envs/e3sm_unified_1.10.0_login/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/global/common/software/e3sm/anaconda_envs/base/envs/e3sm_unified_1.10.0_login/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/global/common/software/e3sm/anaconda_envs/base/envs/e3sm_unified_1.10.0_login/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/global/common/software/e3sm/anaconda_envs/base/envs/e3sm_unified_1.10.0_login/lib/python3.10/site-packages/numpy/lib/nanfunctions.p

[SAVED] /pscratch/sd/z/zhan391/e3sm_dart/diag_dart/fixed_metrics_netcdf/tcc_rmse_metrics_allpct_global_DARTEN40.nc
